In [1]:
import time
import osxphotos

In [2]:
photosdb = osxphotos.PhotosDB('/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary')     # Load the Photos library
# photosdb = osxphotos.PhotosDB()

In [3]:
photos = photosdb.photos()     # Get all photos

In [4]:
movies = photosdb.photos(images=False, movies=True)
images = photosdb.photos(images=True, movies=False)

In [5]:
len(images),len(movies)

(65367, 6240)

In [6]:
def photo_to_basic_row(photo):
    return {
        "uuid": photo.uuid,
        "filename": photo.filename,
        "original_filename": photo.original_filename,
        "path": str(photo.path) if photo.path else None,
        "isphoto": photo.isphoto,
        "ismovie": photo.ismovie,
        "ismissing": photo.ismissing,
        "date": str(photo.date) if photo.date else None,
        "date_added": str(photo.date_added) if photo.date_added else None,
        "title": photo.title,
        "description": photo.description,
        "keywords": photo.keywords,
        "albums": photo.albums,
    }

rows = [photo_to_basic_row(photo) for photo in photos]

len(rows), rows[0]

(71607,
 {'uuid': 'CE32A1E3-C42A-4419-8CC6-60D9ED738581',
  'filename': 'CE32A1E3-C42A-4419-8CC6-60D9ED738581.heic',
  'original_filename': 'IMG_0344.HEIC',
  'path': '/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary/originals/C/CE32A1E3-C42A-4419-8CC6-60D9ED738581.heic',
  'isphoto': True,
  'ismovie': False,
  'ismissing': False,
  'date': '2018-09-11 09:21:22.158427+08:00',
  'date_added': '2018-09-11 09:21:22.574967+08:00',
  'title': None,
  'description': None,
  'keywords': [],
  'albums': []})

In [7]:
hide_photos = [
    photo for photo in photos
    if "HIDE" in photo.keywords
]

len(hide_photos)

23745

In [8]:
for photo in hide_photos[:10]:
    print(photo.uuid)
    print(photo.filename)
    print(photo.date)
    print(photo.keywords)
    print(photo.albums)
    print(photo.path)
    print("-" * 80)

5B1BC36B-B9BF-42BC-B766-29A8D1785A95
5B1BC36B-B9BF-42BC-B766-29A8D1785A95.jpeg
2018-03-08 12:03:57.969425+08:00
['HIDE']
['Girls-2']
/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary/originals/5/5B1BC36B-B9BF-42BC-B766-29A8D1785A95.jpeg
--------------------------------------------------------------------------------
BFE58814-C5AA-440B-AB15-7CF27EA6DA9E
BFE58814-C5AA-440B-AB15-7CF27EA6DA9E.png
2025-01-22 14:58:58.921192+08:00
['HIDE', 'NSFW', 'NSFW_TUMBLR']
['Tumblr Girls - 1']
/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary/originals/B/BFE58814-C5AA-440B-AB15-7CF27EA6DA9E.png
--------------------------------------------------------------------------------
A6E81B3E-DEBA-454A-B757-9B4BF6D1DD14
A6E81B3E-DEBA-454A-B757-9B4BF6D1DD14.png
2023-09-12 10:37:34.978341+08:00
['HIDE', 'NSFW']
['Tumblr Girls - 1']
/Volumes/PRO-G40--20250315/Backup -

In [9]:
p = next(photo for photo in photos if photo.albums)

[x for x in dir(p) if "album" in x.lower() or "folder" in x.lower()]

['_albums',
 '_get_album_uuids',
 'album_info',
 'albums',
 'burst_album_info',
 'burst_albums']

In [10]:
ai = p.album_info[0]

print(type(ai))
print(ai)
print([x for x in dir(ai) if not x.startswith("_")])

<class 'osxphotos.albuminfo.AlbumInfo'>
['asdict', 'creation_date', 'end_date', 'folder_list', 'folder_names', 'library_list_order', 'owner', 'parent', 'photo_index', 'photos', 'sort_order', 'start_date', 'title', 'uuid']


In [11]:
for ai in p.album_info:
    print("album title:", ai.title)
    print("album uuid:", ai.uuid)
    print("folder_names:", ai.folder_names)
    print("parent:", ai.parent)
    print("folder_list:", ai.folder_list)
    print("-" * 80)

album title: Girls-2
album uuid: ECFD948C-8EB6-4573-BEF0-60E58EDD67A4
folder_names: []
parent: None
folder_list: []
--------------------------------------------------------------------------------


In [12]:
def build_library_structure(photos):
    assets = {}
    albums = {}
    folders = {}
    asset_album_links = []

    for photo in photos:
        asset_uuid = photo.uuid

        assets[asset_uuid] = {
            "uuid": photo.uuid,
            "filename": photo.filename,
            "original_filename": photo.original_filename,
            "path": str(photo.path) if photo.path else None,
            "isphoto": photo.isphoto,
            "ismovie": photo.ismovie,
            "ismissing": photo.ismissing,
            "date": str(photo.date) if photo.date else None,
            "date_added": str(photo.date_added) if photo.date_added else None,
            "title": photo.title,
            "description": photo.description,
            "keywords": list(photo.keywords),
            "album_uuids": [],
        }

        for ai in photo.album_info:
            album_uuid = ai.uuid
            folder_path = "/".join(ai.folder_names) if ai.folder_names else ""

            if album_uuid not in albums:
                albums[album_uuid] = {
                    "uuid": album_uuid,
                    "title": ai.title,
                    "folder_names": list(ai.folder_names),
                    "folder_path": folder_path,
                    "asset_uuids": [],
                }

            albums[album_uuid]["asset_uuids"].append(asset_uuid)
            assets[asset_uuid]["album_uuids"].append(album_uuid)

            asset_album_links.append({
                "asset_uuid": asset_uuid,
                "album_uuid": album_uuid,
                "album_title": ai.title,
                "folder_path": folder_path,
            })

            if folder_path:
                if folder_path not in folders:
                    folders[folder_path] = {
                        "folder_path": folder_path,
                        "folder_names": list(ai.folder_names),
                        "album_uuids": set(),
                    }

                folders[folder_path]["album_uuids"].add(album_uuid)

    for folder in folders.values():
        folder["album_uuids"] = list(folder["album_uuids"])

    return {
        "assets": assets,
        "albums": albums,
        "folders": folders,
        "asset_album_links": asset_album_links,
    }

library_structure = build_library_structure(photos)

print("assets:", len(library_structure["assets"]))
print("albums:", len(library_structure["albums"]))
print("folders:", len(library_structure["folders"]))
print("asset_album_links:", len(library_structure["asset_album_links"]))

assets: 71607
albums: 5172
folders: 35
asset_album_links: 65937


In [13]:
for folder_path, folder in sorted(library_structure["folders"].items())[:20]:
    print("FOLDER:", folder_path)
    for album_uuid in folder["album_uuids"]:
        album = library_structure["albums"][album_uuid]
        print("  ALBUM:", album["title"], "assets:", len(album["asset_uuids"]))
    print("-" * 80)

FOLDER: 4G（5G)上網iPhone 12 Pro在溫州街家中測試速度（後面加了捷運跟瑜珈教室5G的測試資料）
  ALBUM: 中華 4G 凌晨五點 家中書房書桌前 assets: 1
  ALBUM: 中華 5G 晚上八點  羅曼羅蘭？（辛亥路五段93號）速度超級快！ assets: 1
  ALBUM: 中華 5G 傍晚 捷運 台北車站剛離站車廂內 assets: 1
  ALBUM: 中華 5G 半夜12點鐘 家中書房書桌前 有時居然收到5G訊號而且很強！速度居然破表的快！但不穩定。一下有5G，一下子又只有4G！然後如果熱點分享居然也可以超過100Mbps assets: 1
  ALBUM: 中華 5G 傍晚 Yoga Edition 6F教室外面飲水機旁邊 assets: 2
  ALBUM: 遠傳 晚上10:30 assets: 5
  ALBUM: 中華 5G 晚上八點 捷運 小南門站月台 assets: 1
  ALBUM: 中華 5G/4G（比較） 晚上八點  出了台電大樓站捷運站的路口 assets: 1
  ALBUM: 遠傳 下午兩點半 assets: 4
  ALBUM: 中華 下午兩點半到三點 assets: 5
  ALBUM: 中華 5G 傍晚 捷運 西門站月台 assets: 1
  ALBUM: 中華 晚上10:30 assets: 8
  ALBUM: 中華 傍晚 5G試用卡（是月租費2399的速度，幾乎沒有參考價值。服務人員還說的一副理所當然，台灣現在年輕人的想法很詭異…這是代溝吧） assets: 2
  ALBUM: 中華 5G 傍晚 捷運 忠孝敦化捷運站月台上 assets: 1
  ALBUM: 中華 5G 傍晚 捷運 西門快到站 車廂內 assets: 1
  ALBUM: 中華 5G 晚上八點 捷運 台北車站往西門剛離站車廂內 assets: 1
  ALBUM: 中華 5G 傍晚 捷運 西門剛離站往台北車站 車廂內 assets: 1
  ALBUM: 中華 傍晚4:30 assets: 2
  ALBUM: 中華 5G 晚上八點 捷運 台電大樓站月台 assets: 1
  ALBUM: 中華 5G 傍晚 捷運 中正紀念堂-》小南門 車廂內 assets: 1
  ALBUM: 中華 5G 凌晨五點 

In [14]:
assets_not_in_album = [
    asset for asset in library_structure["assets"].values()
    if not asset["album_uuids"]
]

len(assets_not_in_album)

7701

In [15]:
albums_not_in_folder = [
    album for album in library_structure["albums"].values()
    if not album["folder_path"]
]

len(albums_not_in_folder), albums_not_in_folder[:5]

(4277,
 [{'uuid': 'ECFD948C-8EB6-4573-BEF0-60E58EDD67A4',
   'title': 'Girls-2',
   'folder_names': [],
   'folder_path': '',
   'asset_uuids': ['5B1BC36B-B9BF-42BC-B766-29A8D1785A95',
    'CE77D672-3626-4886-A231-9B367F2E640E',
    'E4F38046-025F-4067-A37B-4EF0C84393AD',
    '31440F87-0249-47AB-8674-A88A87135B99',
    'DD49F375-375B-424B-B75C-A81C67EA5B00',
    'A7400F96-1FF4-44CA-9640-3F2274BE7710',
    'BF59808A-EDCB-41E5-B131-B5600F1C3B1F',
    'FAFE2C3B-D300-49E3-A482-2725EBEF090A',
    '4BBAA176-68B4-4E0C-9141-9DEC6C530620',
    '59AC7E60-F8D3-4CCF-9EEB-1C3C86FC484A',
    '3C5B8259-3724-4914-9688-78C25E4B1A98',
    'B4182EAB-AD9A-4163-B345-B054A089F7C5',
    '2D92D003-3285-4F87-ACF7-42C08C250903',
    '27FB1C8B-74D4-4EA9-A9A9-1617FB456EE9',
    '5A3A9CF2-E24A-420A-A4C9-6417837D6AAA',
    'EA4BFF64-907C-438B-9063-70AF8C98244C',
    '0236BB47-932A-4773-BF45-23ADA61FEB02',
    '95BB86C3-E55D-49ED-85DC-A1F98573D0A0',
    '1B834482-EB82-4BDA-8409-958C55240EFE',
    '2723A577-8876-4A88

In [16]:
from collections import Counter, defaultdict

folder_counter = Counter()
folder_to_albums = defaultdict(set)

for photo in photos:
    for ai in photo.album_info:
        folder_path = "/".join(ai.folder_names) if ai.folder_names else ""

        if folder_path:
            folder_counter[folder_path] += 1
            folder_to_albums[folder_path].add(ai.title)

print("folders with assets:", len(folder_counter))

for folder_path, count in folder_counter.most_common():
    print(folder_path, "assets:", count, "albums:", len(folder_to_albums[folder_path]))

folders with assets: 35
NSFW assets: 11873 albums: 51
HIDE assets: 8937 albums: 187
股票 assets: 7427 albums: 301
出國旅遊/#北海道便宜團 2024年9月17~9月21日 assets: 1589 albums: 9
開箱 assets: 506 albums: 35
出國旅遊 assets: 497 albums: 2
電器壞掉（不修）/點外送 assets: 466 albums: 38
業障江家/業障江品瑩 assets: 254 albums: 46
業障江家 assets: 246 albums: 29
我 assets: 135 albums: 37
股票/#台股 #重要記事本 assets: 113 albums: 2
NSFW/AV assets: 90 albums: 6
信用卡帳單 assets: 70 albums: 22
重要文件（隱藏） assets: 68 albums: 2
4G（5G)上網iPhone 12 Pro在溫州街家中測試速度（後面加了捷運跟瑜珈教室5G的測試資料） assets: 46 albums: 22
NSFW/SPY_PRETTY_GIRLS assets: 42 albums: 8
Yoga Practice Sequences assets: 39 albums: 3
HIDE_UGLY assets: 35 albums: 9
有趣的東西--雜七雜八 assets: 30 albums: 17
業障江家/林欒菲 assets: 27 albums: 6
我小時候的照片（翻拍） assets: 22 albums: 7
電器壞掉（不修） assets: 21 albums: 3
去澳洲參加1995年國際物理奧林匹亞競賽 assets: 20 albums: 10
新聞 assets: 18 albums: 8
能量冥想＋脈輪瑜伽 線上課程 by Corey assets: 16 albums: 6
股票/#YouTube股市名嘴 #預測 #報明牌 assets: 12 albums: 2
Insta360 assets: 8 albums: 3
送貨 assets: 7 albums: 3
藏傳佛教 as

In [17]:
for folder_path in sorted(folder_to_albums.keys()):
    print("FOLDER:", folder_path)
    for album_title in sorted(folder_to_albums[folder_path]):
        print("  ALBUM:", album_title)
    print("-" * 80)

FOLDER: 4G（5G)上網iPhone 12 Pro在溫州街家中測試速度（後面加了捷運跟瑜珈教室5G的測試資料）
  ALBUM: 中華 4G 凌晨五點 家中書房書桌前
  ALBUM: 中華 5G 傍晚 Yoga Edition 5F到6F樓梯間
  ALBUM: 中華 5G 傍晚 Yoga Edition 6F教室外面飲水機旁邊
  ALBUM: 中華 5G 傍晚 捷運 中正紀念堂-》小南門 車廂內
  ALBUM: 中華 5G 傍晚 捷運 台北車站剛離站車廂內
  ALBUM: 中華 5G 傍晚 捷運 忠孝敦化捷運站月台上
  ALBUM: 中華 5G 傍晚 捷運 西門剛離站往台北車站 車廂內
  ALBUM: 中華 5G 傍晚 捷運 西門快到站 車廂內
  ALBUM: 中華 5G 傍晚 捷運 西門站月台
  ALBUM: 中華 5G 凌晨五點 家中書房書桌前（居然收到5G訊號而且很強！速度居然破表的快！）後來發現不穩定一下有5G，一下子又只有4G！
  ALBUM: 中華 5G 半夜12點鐘 家中書房書桌前 有時居然收到5G訊號而且很強！速度居然破表的快！但不穩定。一下有5G，一下子又只有4G！然後如果熱點分享居然也可以超過100Mbps
  ALBUM: 中華 5G 晚上八點  羅曼羅蘭？（辛亥路五段93號）速度超級快！
  ALBUM: 中華 5G 晚上八點 捷運 台北車站往西門剛離站車廂內
  ALBUM: 中華 5G 晚上八點 捷運 台電大樓站月台
  ALBUM: 中華 5G 晚上八點 捷運 小南門站月台
  ALBUM: 中華 5G/4G（比較） 晚上八點  出了台電大樓站捷運站的路口
  ALBUM: 中華 下午兩點半到三點
  ALBUM: 中華 傍晚 5G試用卡（是月租費2399的速度，幾乎沒有參考價值。服務人員還說的一副理所當然，台灣現在年輕人的想法很詭異…這是代溝吧）
  ALBUM: 中華 傍晚4:30
  ALBUM: 中華 晚上10:30
  ALBUM: 遠傳 下午兩點半
  ALBUM: 遠傳 晚上10:30
--------------------------------------------------------------------------------
FOLDER: Empty Albums
  AL

In [18]:
caption_assets = [
    p for p in photos
    if p.description and str(p.description).strip()
]

print("assets with caption:", len(caption_assets))

assets with caption: 728


In [19]:
for i, p in enumerate(caption_assets, start=1):
    print(f"[{i}]")
    print("uuid:", p.uuid)
    print("filename:", p.filename)
    print("isphoto:", p.isphoto, "ismovie:", p.ismovie)
    print("date:", p.date)
    print("caption:", p.description)
    print("keywords:", p.keywords)
    print("albums:", p.albums)
    print("path:", p.path)
    print("-" * 100)

[1]
uuid: DC0879EA-5308-4915-ACBB-2E788FE5C90B
filename: DC0879EA-5308-4915-ACBB-2E788FE5C90B.png
isphoto: True ismovie: False
date: 2024-05-21 05:59:01+08:00
caption: 2024年5月20日 盤中走勢
keywords: []
albums: ['#台股 #個股 #永昕  #4726 盤中走勢']
path: /Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary/originals/D/DC0879EA-5308-4915-ACBB-2E788FE5C90B.png
----------------------------------------------------------------------------------------------------
[2]
uuid: 60C7418A-6EEB-400B-A8B4-ADCAB30911B7
filename: 60C7418A-6EEB-400B-A8B4-ADCAB30911B7.jpeg
isphoto: True ismovie: False
date: 2025-01-28 20:45:01.452000+08:00
caption: 那我也來加熱不夠燙的佛跳牆
keywords: []
albums: ['#2025年夜飯  2025年1月28日 晚上7:53開始吃。只有我跟爸媽，江品瑩只是來夾菜！而且每次夾菜都擋在我面前，也不會說借過超沒禮貌！這種垃圾人難怪在哪裡都惹人厭！還怪別人要害他還怪上輩子，這輩子個性有問題自己不知道人家跟他說他也聽不進去！年夜飯超沒禮貌！我還沒來吃，它就自己先開動，把魚夾爛了！整個年夜飯我都在爸爸聊天！聊大姑丈和新竹的往事！最後搞到9:50，媽媽才想到他還沒倒垃圾問說幾點鐘！只剩10分鐘垃圾場就要關了，這才匆匆忙忙結束年夜飯。總共聊了大約兩個小時！全程有錄音！垃圾江品瑩應該又是去拉皮，搞得人不像人鬼不像鬼！

In [20]:
favorite_assets = [p for p in photos if p.favorite]
likes_assets = [p for p in photos if p.likes]

print("favorite:", len(favorite_assets))
print("likes:", len(likes_assets))
print("both:", len([p for p in photos if p.favorite and p.likes]))
print("favorite only:", len([p for p in photos if p.favorite and not p.likes]))
print("likes only:", len([p for p in photos if p.likes and not p.favorite]))

favorite: 699
likes: 0
both: 0
favorite only: 699
likes only: 0


In [21]:
favorite_assets = [
    p for p in photos
    if p.favorite
]

print("favorite assets:", len(favorite_assets))

favorite assets: 699


In [22]:
for i, p in enumerate(favorite_assets[:50], start=1):
    print(f"[{i}]")
    print("uuid:", p.uuid)
    print("filename:", p.filename)
    print("isphoto:", p.isphoto)
    print("ismovie:", p.ismovie)
    print("date:", p.date)
    print("favorite:", p.favorite)
    print("keywords:", p.keywords)
    print("caption:", p.description)
    print("albums:", p.albums)
    print()

[1]
uuid: C38E2B63-8D09-491F-93CA-2253D27F4582
filename: C38E2B63-8D09-491F-93CA-2253D27F4582.jpeg
isphoto: True
ismovie: False
date: 2018-07-24 00:41:19.450156+08:00
favorite: True
keywords: ['HIDE']
caption: None
albums: ['Pinterest Girls']

[2]
uuid: 9D40FCAE-9DF1-4758-96AD-309D2A064C42
filename: 9D40FCAE-9DF1-4758-96AD-309D2A064C42.png
isphoto: True
ismovie: False
date: 2024-12-04 09:05:36+08:00
favorite: True
keywords: []
caption: None
albums: ['#台股 #連續吃三隻跌停板 #這輩子沒有這麼無力過  2024年12月4日 這是我做股票的歷史上，值得記錄的一天！第一次吃三隻跌停板跑不掉！開盤前看到超過5000張要又跌停板價格賣，第一盤只有八十幾張買走。最後才一百多張成交！跟被鎖死沒什麼差別！不知道是聽胎盤的時候的手速？還是像股市爆料同學會上面說的拼運氣？']

[3]
uuid: 74744779-9383-49AA-92ED-0F301DA1BA19
filename: 74744779-9383-49AA-92ED-0F301DA1BA19.png
isphoto: True
ismovie: False
date: 2024-07-15 12:06:49+08:00
favorite: True
keywords: []
caption: 後面的三支重電股 本益比都蠻高的，但是在大跌之後居然都可以當沖大漲至少都可以賺一萬元！本益比從47到55甚至到63。 亞力 中興電 士電！ 今天當沖最少也是股價最低的賺$9最多的居然可以賺到$20！
albums: ['#台股 #研究 #盤前盤中操作跟思考 2024年7月15日 上週五大跌之後很緊張，睡覺都睡不好早上醒來五點多在床上就思考今天要怎麼操作要不要賣股票。所以從早上七

In [23]:
# ============================================================
# Explore: filename vs original_filename
# ============================================================

filename_same_count = 0
filename_different_rows = []

for photo in photos:
    filename = photo.filename
    original_filename = photo.original_filename

    if filename == original_filename:
        filename_same_count += 1
    else:
        filename_different_rows.append({
            "uuid": photo.uuid,
            "filename": filename,
            "original_filename": original_filename,
            "path": str(photo.path) if photo.path else None,
            "isphoto": photo.isphoto,
            "ismovie": photo.ismovie,
        })

print("total assets:", len(photos))
print("filename == original_filename:", filename_same_count)
print("filename != original_filename:", len(filename_different_rows))

filename_different_rows[:30]

total assets: 71607
filename == original_filename: 0
filename != original_filename: 71607


[{'uuid': 'CE32A1E3-C42A-4419-8CC6-60D9ED738581',
  'filename': 'CE32A1E3-C42A-4419-8CC6-60D9ED738581.heic',
  'original_filename': 'IMG_0344.HEIC',
  'path': '/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary/originals/C/CE32A1E3-C42A-4419-8CC6-60D9ED738581.heic',
  'isphoto': True,
  'ismovie': False},
 {'uuid': '5B1BC36B-B9BF-42BC-B766-29A8D1785A95',
  'filename': '5B1BC36B-B9BF-42BC-B766-29A8D1785A95.jpeg',
  'original_filename': 'IMG_0579.JPG',
  'path': '/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary/originals/5/5B1BC36B-B9BF-42BC-B766-29A8D1785A95.jpeg',
  'isphoto': True,
  'ismovie': False},
 {'uuid': 'E8CE9C53-A3DC-4102-981A-72136D3773F8',
  'filename': 'E8CE9C53-A3DC-4102-981A-72136D3773F8.heic',
  'original_filename': 'IMG_0323.HEIC',
  'path': '/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20

In [34]:
# ============================================================
# Explore: title vs description/caption
# ============================================================

title_rows = []
caption_rows = []
both_title_and_caption_rows = []

for photo in photos:
    title = photo.title
    caption = photo.description

    has_title = bool(title)
    has_caption = bool(caption)

    if has_title:
        title_rows.append({
            "uuid": photo.uuid,
            "filename": photo.filename,
            "original_filename": photo.original_filename,
            "title": title,
            "caption": caption,
            "title_length": len(title) if title else 0,
            "caption_length": len(caption) if caption else 0,
        })

    if has_caption:
        caption_rows.append({
            "uuid": photo.uuid,
            "filename": photo.filename,
            "original_filename": photo.original_filename,
            "title": title,
            "caption": caption,
            "title_length": len(title) if title else 0,
            "caption_length": len(caption) if caption else 0,
        })

    if has_title and has_caption:
        both_title_and_caption_rows.append({
            "uuid": photo.uuid,
            "filename": photo.filename,
            "original_filename": photo.original_filename,
            "title": title,
            "caption": caption,
            "title_length": len(title) if title else 0,
            "caption_length": len(caption) if caption else 0,
        })

print("total assets:", len(photos))
print("assets with title:", len(title_rows))
print("assets with caption/description:", len(caption_rows))
print("assets with both title and caption:", len(both_title_and_caption_rows))

print("max title length:", max([row["title_length"] for row in title_rows], default=0))
print("max caption length:", max([row["caption_length"] for row in caption_rows], default=0))

total assets: 71607
assets with title: 0
assets with caption/description: 728
assets with both title and caption: 0
max title length: 0
max caption length: 955


In [35]:
# ============================================================
# Explore: caption / description examples
# ============================================================

caption_rows_sorted = sorted(
    caption_rows,
    key=lambda row: row["caption_length"],
    reverse=True,
)

caption_rows_sorted[:30]

[{'uuid': 'D30EE0E9-7134-4355-9EEE-DB88FC1F55B2',
  'filename': 'D30EE0E9-7134-4355-9EEE-DB88FC1F55B2.png',
  'original_filename': 'IMG_0857.PNG',
  'title': None,
  'caption': '2024年8月2日 早上6:50螢幕截圖！7:00在我的日誌上的紀錄如下：忍不住滑YouTube花了22分鐘主要是打開來第一個影片就是一個車展美女。超美的但是沒有名字！我試著用ChatGPT上傳他的照片請他找找不到。在用ChatGPT之前我先用Google圖像搜尋找不到。車子GPT建議我用的TinEye跟Google圖像搜尋一樣，搜尋結果 無！但是用微軟的 Bing搜尋引擎，居然給我一大堆整形臉的韓國車展模特兒…這個就是嚴謹跟不嚴謹的差別。然後韓國人或者是說現代人整形化妝都是一個樣子沒有特色 真的是為難人工智慧了！雖然我們以前2019年人工智慧學校上課的時候確實發現非常小張的圖我們看不出來是什麼東西但是深度學習卻比人員還厲害。現在 照片大張了美肌化妝 這種IG網紅或者是車展美女 反而 我認為 看過的人應該就認得出來 主要是影片有不同角度 就像之前的小龍鼠 有些影片一開始也看不出來但是他的特徵就是長得像瑜伽老師YaYa 這部分 在觀看影片之後就可以找得出來。所以這一波的人工智慧是到頭了嗎？深度學習跟Google的 Transformer 「transformer is all you need」這整套技術就只能達到這個程度嗎？又達到這一波A I的極限了嗎？那麼A I這個不是泡沫的泡沫要破了嗎？ VR的泡沫早就破掉了！ A I確實是個泡沫因為並沒有落地也就是說連我爸爸媽媽臭雞巴江品瑩都在使用這時候才叫iPhone等級的新科技！自從我對Apple Intelligent能夠做的事情破滅之後，我對A I又不看好了！當真是回歸初心啊（亂用詞語。去年做股票九月10月的時候我就不看好A I 因為我懂 不過我懂的是2019年的人工智慧技術。根本不能用！一直到2021-2022我好像都還有參加比賽什麼芒果圖像辨識的比賽。2023年初ChatGPT出來 我馬上就用了一下但是覺得普普通通所以才會覺得A I還是那副鳥樣。但是大家都炒得火熱。

In [36]:
# ============================================================
# Explore: album title length and diary-like album titles
# ============================================================

album_rows_by_key = {}

for photo in photos:
    for album_info in photo.album_info:
        folder_names = list(album_info.folder_names)
        folder_path = "/".join(folder_names) if folder_names else ""
        album_title = album_info.title
        album_uuid = album_info.uuid

        album_key = (folder_path, album_title, album_uuid)

        if album_key not in album_rows_by_key:
            album_rows_by_key[album_key] = {
                "album_uuid": album_uuid,
                "folder_path": folder_path,
                "folder_names": folder_names,
                "album_title": album_title,
                "album_title_length": len(album_title) if album_title else 0,
                "asset_count_seen_from_album_info": 0,
            }

        album_rows_by_key[album_key]["asset_count_seen_from_album_info"] += 1

album_rows = list(album_rows_by_key.values())

album_rows_sorted = sorted(
    album_rows,
    key=lambda row: row["album_title_length"],
    reverse=True,
)

print("unique album rows:", len(album_rows_sorted))
print("max album title length:", album_rows_sorted[0]["album_title_length"] if album_rows_sorted else 0)

album_rows_sorted[:30]

unique album rows: 5172
max album title length: 2809


[{'album_uuid': 'EBEA07A0-500F-45E6-94D4-77BA5726526E',
  'folder_path': '股票',
  'folder_names': ['股票'],
  'album_title': '#台股 #賣股 #大跌 #回檔修正 2024年7月12日 因為昨天只睡兩個小時，又是作息不正常不舒服的狀況下等開盤。8:22 我就來看剩下的兩隻個股：順藥 跟 永信建 ！ 開盤前就都給他們漲停板的價格賣出！因為這兩支很有可能在台股大跌的時候逆勢上漲！如果我睡著沒有賣到就虧了特別是 永信建！早就該獲利了結！星期二就是睡覺耽誤我賣$226（今天最後賣在$220 賠了$6000 … …幹！睡覺誤事！） 順藥 也是沒注意， 7月4日$244 ，我是$226買的，沒有注意7月4號處置結束，外資會直接大賣999張股票！前一天如果就賣的話，就直接賺$18,000！我真的不會做！正在慢慢學！ 結果因為下跌不甘心手上有錢結果加碼在2百21.5 ＄但是那天卻一路跌到$217 ，現在只剩$208.5！假設7月3號賣掉賺了$18,000 ，現在又可以低價買入而且不會買第二張！一步錯步步錯！回到今天的操作跟過程中的心理狀態。8:49又開始收集開盤前試撮的資料做螢幕截圖。看到一片都是綠的只有幾個紅的！居然還有紅的！順藥他媽的騙人是紅色但卻以下跌收盤！跌了$0.5。 8:50看到今天應該要大跌但是 上詮 在試撮的時候居然漲停板！搞什麼鬼？！這些妖怪！9:01開盤我就去看 美時 ， 因為他昨天除息又下跌，我就看是不是好的買一點。沒想到開盤上漲，我在操作的過程中來來回回一直看他，在我休息睡覺前都不覺得是好的買一點。雖然最後收盤是下跌$1！不過後面下跌的時候我已經去睡覺了，沒有機會買。而且他的價格有點貴要大概28萬元，買下去基本上就沒有現金了！不過如果會快速大漲，這就不是什麼問題。昨天除息完又繼續大跌，今天在幾乎所有均線下方。遠低於半年線！下面只剩下一條年線！反正沒買明天看看狀況，增加我做短線的經驗值！9:07注意力回到 永信建。當時價格$222最高價格有到$222.5不過才開盤10分鐘所以我先樂觀一點改價格以$223.5委托賣出。沒有幾分鐘他就下跌了跌到$220！因為很累精神不好，不想玩了！而且已經有賺錢，我9:14就以線價$220賣出直接成交。買的時候因為

In [37]:
# ============================================================
# Explore: folders inferred from album_info.folder_names
# ============================================================

folder_paths = set()
folder_examples = {}

for photo in photos:
    for album_info in photo.album_info:
        folder_names = list(album_info.folder_names)

        for i in range(1, len(folder_names) + 1):
            folder_path = "/".join(folder_names[:i])
            folder_paths.add(folder_path)

            if folder_path not in folder_examples:
                folder_examples[folder_path] = {
                    "folder_path": folder_path,
                    "folder_names_prefix": folder_names[:i],
                    "example_album_title": album_info.title,
                    "example_asset_filename": photo.filename,
                }

folder_rows = [
    folder_examples[folder_path]
    for folder_path in sorted(folder_paths)
]

print("folder count inferred from album_info.folder_names:", len(folder_rows))

folder_rows[:50]

folder count inferred from album_info.folder_names: 35


[{'folder_path': '4G（5G)上網iPhone 12 Pro在溫州街家中測試速度（後面加了捷運跟瑜珈教室5G的測試資料）',
  'folder_names_prefix': ['4G（5G)上網iPhone 12 Pro在溫州街家中測試速度（後面加了捷運跟瑜珈教室5G的測試資料）'],
  'example_album_title': '中華 5G 晚上八點 捷運 小南門站月台',
  'example_asset_filename': '99C017AD-4F89-4DBF-816E-89B0EA73F7B2.mp4'},
 {'folder_path': 'Empty Albums',
  'folder_names_prefix': ['Empty Albums'],
  'example_album_title': '#給資料夾置頂用',
  'example_asset_filename': '29E1E243-AEAF-43D7-BF36-AD3B4C0FF850.png'},
 {'folder_path': 'HIDE',
  'folder_names_prefix': ['HIDE'],
  'example_album_title': '妖狐 girls',
  'example_asset_filename': 'DC6B3BC2-84A7-4707-B43E-A034BAD34ADF.png'},
 {'folder_path': 'HIDE_UGLY',
  'folder_names_prefix': ['HIDE_UGLY'],
  'example_album_title': '（隱藏內容，醜八怪，醜人多作怪，看了噁心）2024年2月5日晚上7:42 發現下午行蹤鬼祟的臭雞巴兩人回家了。之前也被我拆穿畜生江去整容，只是因為桌上有一大堆蜈蚣草煎好的水間要， Google發現這個是醫美整容之後消腫用的！總之我想知道到底是怎樣見不得人，就把我的iPhone用超廣角放在門口地上堵住在廁所的畜生江品瑩！不過為了一定排到沒有用望遠鏡頭畫面非常不清楚。但是還是可以明顯看出臉上沒有包紗布，這麼不清楚都可以看出來還是很醜所以應該是無效的整容！浪費錢還要浪費大家的時間！可笑！2月6日 畜生三人 三點鐘就不見了，結果我去上完第一堂瑜伽

In [38]:
# ============================================================
# Explore: keywords distribution and HIDE usage
# ============================================================

from collections import Counter

keyword_counter = Counter()
assets_with_keywords = []
assets_with_HIDE = []

for photo in photos:
    keywords = list(photo.keywords)

    if keywords:
        assets_with_keywords.append({
            "uuid": photo.uuid,
            "filename": photo.filename,
            "original_filename": photo.original_filename,
            "keywords": keywords,
            "favorite": photo.favorite,
            "caption": photo.description,
            "albums": photo.albums,
        })

    for keyword in keywords:
        keyword_counter[keyword] += 1

    if "HIDE" in keywords:
        assets_with_HIDE.append({
            "uuid": photo.uuid,
            "filename": photo.filename,
            "original_filename": photo.original_filename,
            "keywords": keywords,
            "favorite": photo.favorite,
            "caption": photo.description,
            "albums": photo.albums,
        })

print("total assets:", len(photos))
print("assets with any keywords:", len(assets_with_keywords))
print("assets with HIDE:", len(assets_with_HIDE))
print("unique keywords:", len(keyword_counter))

keyword_counter.most_common(30)

total assets: 71607
assets with any keywords: 23758
assets with HIDE: 23745
unique keywords: 38


[('HIDE', 23745),
 ('NSFW', 11600),
 ('NSFW_IG', 1594),
 ('NSFW_TUMBLR', 1080),
 ('HIDE_TEMP', 484),
 ('NSFW-LINE', 306),
 ('NSFW-AV', 165),
 ('HIDE_UGLY_SISTER', 158),
 ('HIDE_SIN_CHIANG_FAMILY', 114),
 ('NSFW-AI', 91),
 ('HIDE_IMPORTANT_DOC', 68),
 ('HIDE_DOC', 64),
 ('NSFW-SELFIE-AV', 53),
 ('NSFW-SPY', 49),
 ('HIDE_UGLY', 42),
 ('HIDE_BITCH_FANNY', 42),
 ('SPY_PRETTY_GIRLS', 41),
 ('SECRETS_SISTER', 39),
 ('NSFW-SM', 21),
 ('NSFW-BOMI', 14),
 ('HIDE_UGLY_EX', 8),
 ('HIDE_UGLY_ME', 6),
 ('HIDE_UGLY_DAD', 5),
 ('FAILED_NOSE_JOB', 4),
 ('HIDE_TEMP_DOC', 4),
 ('KEY-001', 2),
 ('NSFW_SEARCH', 2),
 ('Checkpoint-20200701054442', 1),
 ('Checkpoint-20180426001421', 1),
 ('Checkpoint-20190703110755', 1)]

In [39]:
# ============================================================
# Explore: favorite / likes sanity check
# ============================================================

favorite_rows = []
likes_rows = []
both_favorite_and_likes_rows = []

for photo in photos:
    if photo.favorite:
        favorite_rows.append({
            "uuid": photo.uuid,
            "filename": photo.filename,
            "original_filename": photo.original_filename,
            "favorite": photo.favorite,
            "likes": photo.likes,
            "caption": photo.description,
            "keywords": list(photo.keywords),
            "albums": photo.albums,
        })

    if photo.likes:
        likes_rows.append({
            "uuid": photo.uuid,
            "filename": photo.filename,
            "original_filename": photo.original_filename,
            "favorite": photo.favorite,
            "likes": photo.likes,
            "caption": photo.description,
            "keywords": list(photo.keywords),
            "albums": photo.albums,
        })

    if photo.favorite and photo.likes:
        both_favorite_and_likes_rows.append({
            "uuid": photo.uuid,
            "filename": photo.filename,
            "original_filename": photo.original_filename,
            "favorite": photo.favorite,
            "likes": photo.likes,
            "caption": photo.description,
            "keywords": list(photo.keywords),
            "albums": photo.albums,
        })

print("favorite count:", len(favorite_rows))
print("likes count:", len(likes_rows))
print("both favorite and likes count:", len(both_favorite_and_likes_rows))

favorite_rows[:30]

favorite count: 699
likes count: 0
both favorite and likes count: 0


[{'uuid': 'C38E2B63-8D09-491F-93CA-2253D27F4582',
  'filename': 'C38E2B63-8D09-491F-93CA-2253D27F4582.jpeg',
  'original_filename': 'IMG_0082.JPG',
  'favorite': True,
  'likes': [],
  'caption': None,
  'keywords': ['HIDE'],
  'albums': ['Pinterest Girls']},
 {'uuid': '9D40FCAE-9DF1-4758-96AD-309D2A064C42',
  'filename': '9D40FCAE-9DF1-4758-96AD-309D2A064C42.png',
  'original_filename': 'IMG_1103.PNG',
  'favorite': True,
  'likes': [],
  'caption': None,
  'keywords': [],
  'albums': ['#台股 #連續吃三隻跌停板 #這輩子沒有這麼無力過  2024年12月4日 這是我做股票的歷史上，值得記錄的一天！第一次吃三隻跌停板跑不掉！開盤前看到超過5000張要又跌停板價格賣，第一盤只有八十幾張買走。最後才一百多張成交！跟被鎖死沒什麼差別！不知道是聽胎盤的時候的手速？還是像股市爆料同學會上面說的拼運氣？']},
 {'uuid': '74744779-9383-49AA-92ED-0F301DA1BA19',
  'filename': '74744779-9383-49AA-92ED-0F301DA1BA19.png',
  'original_filename': 'IMG_1690.PNG',
  'favorite': True,
  'likes': [],
  'caption': '後面的三支重電股 本益比都蠻高的，但是在大跌之後居然都可以當沖大漲至少都可以賺一萬元！本益比從47到55甚至到63。 亞力 中興電 士電！ 今天當沖最少也是股價最低的賺$9最多的居然可以賺到$20！',
  'keywords': [],
  'albums': ['#台股 #研究 #盤前盤中操作跟思

In [24]:
# missing_photos = []
# downloaded_photos = []
# for photo in photos:
#     if photo.ismissing:
#         missing_photos.append(photo)
#     else:
#         downloaded_photos.append(photo)

In [25]:
# len(missing_photos)

In [26]:
# len(downloaded_photos)

In [27]:
# missing_photos.sort(key=lambda x: x.date_added, reverse=True)
# downloaded_photos.sort(key=lambda x: x.date_added, reverse=True)

In [28]:
# ["{}".format(x.date_added) for x in missing_photos]

In [29]:
# ["{}".format(x.date_added) for x in downloaded_photos]

In [30]:
# for photo in photos:
#     if "{}".format(photo.date_added) == '2025-01-15 15:38:11.807182+08:00':
#         print("Photo found: {}".format(photo))
#         break

In [31]:
# if len(missing_photos)>0:
#     print(missing_photos[0])

In [32]:
# photos[0]

In [33]:
# photos[0].path